# Fragrantica Accord Analysis

## Objective

Analyze the prevalence, distribution, and rating patterns of fragrance accords in the cleaned Fragrantica reference dataset. The analysis examines the most common accords, differences across gender categories, and rating variations relative to gender-specific baselines.

The objective is to identify meaningful patterns in accord prevalence and fragrance ratings and prepare structured analytical results for visualization and further exploration.

## Data Source

- **Source table:** `main.fragrance_analytics.fragrantica_perfumes`
- **Format:** Delta table
- **Platform:** Databricks

The source table contains cleaned fragrance records, including fragrance names, standardized gender classifications (`women`, `men`, and `unisex`), ratings, main accords, perfumers, descriptions, and source URLs.

## Workflow

1. Inspect the structure and format of the `main_accords` column.
2. Parse accord lists stored as strings into individual accord values.
3. Explode the lists to create a fragrance–accord analytical dataset.
4. Validate the parsed data and identify missing or invalid accord values.
5. Analyze overall accord frequency and prevalence.
6. Compare accord prevalence across gender categories.
7. Calculate average and median ratings for accords within each gender category.
8. Compare accord ratings against gender-specific rating baselines.
9. Identify the highest-rated accords and the largest positive and negative rating differences, applying a minimum sample-size threshold.
10. Prepare analytical results for visualization and interpretation.

## Expected Outputs

- A fragrance–accord dataset containing individual fragrance–accord combinations.
- Overall accord frequency and prevalence metrics.
- Gender-specific accord prevalence comparisons.
- Accord-level rating statistics, including average and median ratings.
- Comparisons of accord ratings against gender-specific baselines.
- Ranked results highlighting highly rated accords and accords with the largest rating differences.

These outputs provide a foundation for fragrance analytics, data visualization, and potential natural-language exploration through Databricks AI/BI Genie.

**Interpretation note:** Rating differences and accord prevalence represent observed associations in the dataset. They do not establish causation or directly measure consumer preferences.

## Technologies

Databricks · Python · PySpark · SQL · Delta Lake

In [0]:

from pyspark.sql import functions as F

# Load the cleaned Fragrantica reference table
df_perfumes = spark.table(
    "main.fragrance_analytics.fragrantica_perfumes"
)

# Inspect the schema
df_perfumes.printSchema()

# Check total records and missing accord values
df_perfumes.select(
    F.count("*").alias("total_records"),
    F.sum(
        F.when(
            F.col("main_accords").isNull()
            | (F.trim(F.col("main_accords")) == ""),
            1
        ).otherwise(0)
    ).alias("missing_accords")
).show()

# Display sample accord values
df_perfumes.select(
    "name",
    "gender",
    "main_accords"
).show(20, truncate=False)


In [0]:

from pyspark.sql import functions as F

# Parse the string representation of accord lists into arrays
df_accords = df_perfumes.withColumn(
    "accords",
    F.from_json(
        F.regexp_replace(
            F.regexp_replace(
                F.col("main_accords"),
                "'", '"'
            ),
            r"\bNone\b", "null"
        ),
        "array<string>"
    )
)

# Explode the array: one row per fragrance–accord combination
df_fragrance_accords = (
    df_accords
    .select(
        "id",
        "name",
        "gender",
        "rating_value",
        "rating_count",
        F.explode("accords").alias("accord")
    )
    .withColumn("accord", F.trim(F.lower(F.col("accord"))))
    .filter(F.col("accord").isNotNull() & (F.col("accord") != ""))
)

# Validate the result
print("Original fragrance records:", df_perfumes.count())
print("Fragrance–accord rows:", df_fragrance_accords.count())
print(
    "Distinct accords:",
    df_fragrance_accords.select("accord").distinct().count()
)

display(df_fragrance_accords.limit(20))


In [0]:

from pyspark.sql import functions as F

# 1. Check how many records produced a valid, non-empty accord array
df_accords.select(
    F.count("*").alias("total_fragrances"),
    F.sum(
        F.when(F.col("accords").isNull(), 1).otherwise(0)
    ).alias("null_accord_arrays"),
    F.sum(
        F.when(
            F.col("accords").isNotNull()
            & (F.size("accords") == 0),
            1
        ).otherwise(0)
    ).alias("empty_accord_arrays")
).show()

# 2. Check whether any fragrance has duplicate accord entries
duplicate_accords = (
    df_fragrance_accords
    .groupBy("id", "accord")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate fragrance–accord combinations:",
    duplicate_accords.count()
)

# 3. Summarize the number of accords per fragrance
accords_per_fragrance = (
    df_fragrance_accords
    .groupBy("id")
    .agg(F.countDistinct("accord").alias("accord_count"))
)

accords_per_fragrance.select(
    F.min("accord_count").alias("min_accords"),
    F.avg("accord_count").alias("avg_accords"),
    F.max("accord_count").alias("max_accords")
).show()


In [0]:

from pyspark.sql import functions as F

# Count distinct accord labels at each stage
print(
    "Distinct non-empty accord strings in the original arrays:",
    df_accords
    .select(F.explode("accords").alias("accord"))
    .filter(F.col("accord").isNotNull() & (F.trim("accord") != ""))
    .select("accord")
    .distinct()
    .count()
)

print(
    "Distinct normalized accords after exploding:",
    df_fragrance_accords.select("accord").distinct().count()
)

# Show all distinct accord labels from the exploded dataset
display(
    df_fragrance_accords
    .select("accord")
    .distinct()
    .orderBy("accord")
)


In [0]:

from pyspark.sql import functions as F

# Count fragrances containing each accord
accord_prevalence = (
    df_fragrance_accords
    .groupBy("accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count")
    )
)

# Denominator: fragrances with at least one accord
fragrances_with_accords = (
    df_fragrance_accords
    .select("id")
    .distinct()
    .count()
)

# Calculate prevalence
accord_prevalence = (
    accord_prevalence
    .withColumn(
        "prevalence_pct",
        F.round(
            F.col("fragrance_count")
            / F.lit(fragrances_with_accords) * 100,
            2
        )
    )
)

# Calculate average rating and number of rated fragrances per accord
accord_ratings = (
    df_fragrance_accords.alias("a")
    .join(
        df_perfumes.select("id", "rating_value").alias("p"),
        on="id",
        how="left"
    )
    .groupBy("accord")
    .agg(
        F.countDistinct(
            F.when(F.col("p.rating_value").isNotNull(), F.col("a.id"))
            ).alias("rated_fragrance_count"),
        F.round(F.avg("p.rating_value"), 3).alias("avg_rating")
    )
)

# Combine prevalence and rating metrics
accord_prevalence = (
    accord_prevalence
    .join(accord_ratings, on="accord", how="left")
    .orderBy(F.desc("fragrance_count"), F.asc("accord"))
)

print("Fragrances with at least one accord:", fragrances_with_accords)

display(accord_prevalence)


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
gender_counts = (
    df_fragrance_accords
    .select("id", "gender")
    .distinct()
    .groupBy("gender")
    .agg(
        F.countDistinct("id").alias("fragrance_count")
    )
    .orderBy(F.desc("fragrance_count"))
)

display(gender_counts)

# Count distinct fragrances in each gender group
gender_denominators = (
    df_fragrance_accords
    .select("id", "gender")
    .distinct()
    .groupBy("gender")
    .agg(
        F.countDistinct("id").alias("gender_fragrance_count")
    )
)

# Count fragrances containing each accord within each gender group
accords_by_gender = (
    df_fragrance_accords
    .groupBy("gender", "accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count")
    )
    .join(gender_denominators, on="gender", how="inner")
    .withColumn(
        "prevalence_pct",
        F.round(
            F.col("fragrance_count")
            / F.col("gender_fragrance_count") * 100,
            2
        )
    )
    .orderBy("gender", F.desc("prevalence_pct"), F.asc("accord"))
)

display(accords_by_gender)

Databricks visualization. Run in Databricks to view.

In [0]:

# Compare accord prevalence across gender groups
accord_gender_comparison = (
    accords_by_gender
    .groupBy("accord")
    .agg(
        F.coalesce(
            F.max(
                F.when(
                    F.col("gender") == "men",
                    F.col("prevalence_pct")
                )
            ),
            F.lit(0.0)
        ).alias("men_pct"),

        F.coalesce(
            F.max(
                F.when(
                    F.col("gender") == "women",
                    F.col("prevalence_pct")
                )
            ),
            F.lit(0.0)
        ).alias("women_pct"),

        F.coalesce(
            F.max(
                F.when(
                    F.col("gender") == "unisex",
                    F.col("prevalence_pct")
                )
            ),
            F.lit(0.0)
        ).alias("unisex_pct")
    )
    .withColumn(
        "max_prevalence_pct",
        F.greatest("men_pct", "women_pct", "unisex_pct")
    )
    .withColumn(
        "min_prevalence_pct",
        F.least("men_pct", "women_pct", "unisex_pct")
    )
    .withColumn(
        "gap_pp",
        F.round(
            F.col("max_prevalence_pct")
            - F.col("min_prevalence_pct"),
            2
        )
    )
    .orderBy(F.desc("gap_pp"), F.asc("accord"))
)

display(accord_gender_comparison)


Databricks visualization. Run in Databricks to view.

In [0]:

# Calculate overall prevalence for each accord
total_fragrances_with_accords = (
    df_fragrance_accords
    .select("id")
    .distinct()
    .count()
)

overall_accord_prevalence = (
    df_fragrance_accords
    .groupBy("accord")
    .agg(
        F.countDistinct("id").alias("overall_fragrance_count")
    )
    .withColumn(
        "overall_prevalence_pct",
        F.round(
            F.col("overall_fragrance_count")
            / F.lit(total_fragrances_with_accords) * 100,
            2
        )
    )
)

# Focus on commonly occurring accords with differences across groups
common_accord_comparison = (
    accord_gender_comparison
    .join(overall_accord_prevalence, on="accord", how="left")
    .filter(F.col("overall_prevalence_pct") >= 10)
    .select(
        "accord",
        "overall_fragrance_count",
        "overall_prevalence_pct",
        "men_pct",
        "women_pct",
        "unisex_pct",
        "gap_pp"
    )
    .orderBy(F.desc("gap_pp"))
)

display(common_accord_comparison)


Databricks visualization. Run in Databricks to view.

In [0]:

# One record per fragrance, including its rating and gender
fragrance_ratings = (
    df_perfumes
    .select(
        "id",
        "gender",
        "rating_value",
        "rating_count"
    )
    .dropDuplicates(["id"])
)

# Aggregate ratings for each accord and gender group
accord_rating_by_gender = (
    df_fragrance_accords
    .select("id", "gender", "accord")
    .distinct()
    .join(
        fragrance_ratings.select(
            "id", "rating_value", "rating_count"
        ),
        on="id",
        how="left"
    )
    .groupBy("gender", "accord")
    .agg(
        F.countDistinct("id").alias("fragrance_count"),
        F.count("rating_value").alias("rated_fragrance_count"),
        F.round(F.avg("rating_value"), 3).alias("avg_rating"),
        F.percentile_approx(
            "rating_value", 0.5, 10000
        ).alias("median_rating"),
        F.sum("rating_count").alias("total_rating_count"),
        F.round(F.avg("rating_count"), 1).alias("avg_rating_count")
    )
    .orderBy("gender", F.desc("avg_rating"))
)

display(accord_rating_by_gender)

In [0]:

from pyspark.sql import functions as F
from pyspark.sql import Window

# Minimum sample size for reliable comparison
MIN_RATED_FRAGRANCES = 100

# 1. Calculate the overall rating baseline for each gender
gender_rating_baseline = (
    fragrance_ratings
    .groupBy("gender")
    .agg(
        F.count("rating_value").alias("baseline_rated_count"),
        F.round(F.avg("rating_value"), 3).alias("gender_avg_rating"),
        F.round(
            F.percentile_approx("rating_value", 0.5, 10000), 3
        ).alias("gender_median_rating")
    )
)

# 2. Prepare one shared dataset of accord ratings and gender baselines
accord_rating_analysis = (
    accord_rating_by_gender
    .filter(F.col("rated_fragrance_count") >= MIN_RATED_FRAGRANCES)
    .join(gender_rating_baseline, on="gender", how="left")
    .withColumn(
        "rating_difference",
        F.round(F.col("avg_rating") - F.col("gender_avg_rating"), 3)
    )
)

# 3. Rank accords by average rating within each gender
rating_window = (
    Window
    .partitionBy("gender")
    .orderBy(
        F.desc("avg_rating"),
        F.desc("median_rating"),
        F.desc("rated_fragrance_count")
    )
)

top_rated_accords = (
    accord_rating_analysis
    .withColumn("rank", F.row_number().over(rating_window))
    .filter(F.col("rank") <= 10)
    .select(
        "gender",
        "rank",
        "accord",
        "rated_fragrance_count",
        "avg_rating",
        "median_rating",
        "avg_rating_count"
    )
    .orderBy("gender", "rank")
)

# 4. Compare each accord's average rating with its gender baseline
accord_rating_vs_baseline = (
    accord_rating_analysis
    .select(
        "gender",
        "accord",
        "rated_fragrance_count",
        "avg_rating",
        "gender_avg_rating",
        "rating_difference",
        "median_rating",
        "gender_median_rating"
    )
    .orderBy("gender", F.desc("rating_difference"))
)

# 5. Display both analytical views
print("Top 10 highest-rated accords within each gender")
display(top_rated_accords)

print("Rating differences relative to each gender's average")
display(accord_rating_vs_baseline)


In [0]:

# Rank accords by rating difference within each gender
rating_difference_window = (
    Window
    .partitionBy("gender")
    .orderBy(F.desc("rating_difference"), F.asc("accord"))
)

rating_difference_ranked = (
    accord_rating_vs_baseline
    .withColumn(
        "positive_rank",
        F.row_number().over(rating_difference_window)
    )
)

# Top 10 accords above the gender baseline
top_above_baseline = (
    rating_difference_ranked
    .filter(F.col("positive_rank") <= 10)
    .withColumn("comparison", F.lit("Above baseline"))
)

# Bottom 10 accords below the gender baseline
negative_window = (
    Window
    .partitionBy("gender")
    .orderBy(F.asc("rating_difference"), F.asc("accord"))
)

bottom_below_baseline = (
    accord_rating_vs_baseline
    .withColumn(
        "negative_rank",
        F.row_number().over(negative_window)
    )
    .filter(F.col("negative_rank") <= 10)
    .withColumn("comparison", F.lit("Below baseline"))
)

# Combine both views
rating_extremes = (
    top_above_baseline
    .select(
        "gender", "comparison", "accord",
        "rated_fragrance_count", "avg_rating",
        "gender_avg_rating", "rating_difference"
    )
    .unionByName(
        bottom_below_baseline.select(
            "gender", "comparison", "accord",
            "rated_fragrance_count", "avg_rating",
            "gender_avg_rating", "rating_difference"
        )
    )
    .orderBy(
        "gender",
        "comparison",
        F.desc("rating_difference")
    )
)

display(rating_extremes)

## 9. Key Findings and Interpretation

### 9.1. Overall Accord Distribution

Woody is the most prevalent accord in the Fragrantica dataset, appearing in approximately 65.8% of fragrances with available accord information. Powdery (51.8%), citrus (49.1%), sweet (47.3%), and aromatic (43.1%) are also widespread.

These results indicate that a relatively small group of accords is prominent across the catalogue. However, accords are not mutually exclusive: a fragrance can have multiple accords, so prevalence percentages are not expected to sum to 100%.

### 9.2. Differences Across Gender Categories

Accord prevalence varies considerably across fragrances categorized for men, women, and unisex audiences. The largest differences are observed for aromatic, fresh spicy, floral, and white floral accords.

**Aromatic and fresh spicy accords** are particularly prevalent in men's fragrances. Aromatic appears in 77.84% of men's fragrances, compared with 45.81% of unisex and 25.15% of women's fragrances. Fresh spicy follows a similar pattern, at 66.83%, 38.31%, and 20.36%, respectively.

**Floral and white floral accords** show the opposite pattern. Floral appears in 59.38% of women's fragrances, compared with 30.70% of unisex and 14.82% of men's fragrances. White floral appears in 48.79% of women's fragrances, compared with 25.56% of unisex and 14.74% of men's fragrances.

**Sweet and fruity accords** are also more prevalent in women's fragrances. Sweet appears in 59.81% of women's fragrances versus 28.43% of men's fragrances, while fruity appears in 51.16% versus 19.72%.

**Woody accords are widespread across all three categories**, appearing in 85.04% of men's fragrances, 66.39% of unisex fragrances, and 56.91% of women's fragrances. Despite their broad representation, woody accords show a 28.13-percentage-point gap between the highest and lowest category.

The largest overall prevalence gap is for aromatic accords (52.69 percentage points), followed by fresh spicy (46.47), floral (44.56), and white floral (34.05). These gaps represent the difference between the highest and lowest prevalence among the three categories, rather than necessarily the difference between men's and women's fragrances.

By contrast, some accords have relatively similar prevalence across categories. Musky has a gap of 7.01 percentage points, while salty, anis, and aldehydic have gaps below 2 percentage points. Very rare accords should be interpreted cautiously, since small percentage differences do not necessarily indicate meaningful market patterns.

These results describe the distribution of accord tags across Fragrantica's gender categories. They do not establish that particular accords are inherently gendered or directly measure consumer preferences.

### 9.3. Community Ratings and Accord Associations

Average community ratings across the three gender categories are relatively similar: 3.975 for men's fragrances, 4.004 for unisex fragrances, and 3.949 for women's fragrances.

Comparing each accord's average rating with the corresponding gender-category baseline reveals several recurring patterns.

**Accords associated with above-baseline ratings** include oud, honey, and cinnamon. Oud has an average rating of 4.115 among men's fragrances and 4.091 among unisex fragrances, corresponding to differences of +0.140 and +0.087 rating points from their respective category baselines. Honey is particularly notable among men's fragrances, with an average rating of 4.138, or +0.163 points above baseline. Cinnamon is associated with above-baseline ratings among both unisex (+0.075) and women's fragrances (+0.092).

**Accords associated with below-baseline ratings** include aquatic, marine, and salty, which appear among the below-baseline results across all three categories. The largest negative differences in the analyzed results occur for mineral and cannabis accords in unisex fragrances, with average ratings of 3.758 and 3.761, respectively. These correspond to differences of −0.246 and −0.243 points from the unisex baseline.

The magnitude of these differences varies. Many accords differ from their category baseline by only a few hundredths of a rating point, while some show larger deviations. The number of rated fragrances also varies by accord, so the results should be interpreted alongside sample sizes.

These patterns are associations rather than evidence of causation. A fragrance can contain multiple accords, and its rating may depend on factors such as brand, price, popularity, composition, and the characteristics of the people who rate it. The analysis does not establish that a particular accord directly increases or decreases a fragrance's rating, nor does it demonstrate statistical significance.

The results identify accords worth exploring in further analysis and may inform future recommendation research. However, a recommendation system would need to be validated against individual preferences and additional product-level factors before these associations could be used reliably.

### 9.4. Limitations

- **Catalogue coverage:** The dataset represents fragrances listed on Fragrantica, not the entire fragrance market.
- **Gender classification:** The categories reflect catalogue classifications and may not represent how all consumers use or perceive fragrances.
- **Accord prevalence:** Prevalence measures the proportion of fragrances tagged with an accord, not its intensity or importance to consumers.
- **Community ratings:** Ratings may be influenced by product popularity, rating volume, audience composition, brand, and other factors.
- **Rare accords:** Results for accords appearing in relatively few fragrances may be less stable or of limited analytical significance.
- **Overlapping accords:** A fragrance can contain multiple accords, so rating associations for different accords are not independent.
- **Observational analysis:** The analysis identifies descriptive patterns and associations but does not establish causal relationships or statistical significance.

### 9.5. Conclusion

This analysis establishes a descriptive baseline for exploring the structure of fragrance accords in the Fragrantica catalogue. It identifies the most prevalent accords, highlights differences in their distribution across gender categories, and examines how average community ratings for fragrances containing particular accords compare with their category baselines.

The findings reveal clear differences in accord representation across catalogue categories, alongside potential associations between certain accords and higher or lower average ratings. These observations can support further fragrance analytics projects, including product segmentation and accord-based fragrance recommendations.

Any recommendation system developed from these findings would require additional validation to establish whether catalogue accord profiles and rating associations meaningfully reflect individual user preferences.